# TechMart Customer Service Assistant
## RAG + MCP Integration with Llama Stack

This notebook demonstrates:
1. **RAG (Retrieval-Augmented Generation)**: Using Llama Stack's vector store and file_search tool to retrieve return policy information
2. **MCP (Model Context Protocol)**: Using Llama Stack's native MCP integration to access real-time order data
3. **Integration**: Combining both to answer customer service questions intelligently

## 1. Setup and Installation

In [ ]:
# Install required packages
%pip install llama_stack_client rich

In [ ]:
from llama_stack_client import LlamaStackClient
import rich
import json

## 2. Environment variables

In [ ]:
# Configuration
LLAMA_STACK_URL = "http://techmart-llama-stack-service.llama.svc.cluster.local:8321"
MCP_SERVER_URL = "http://techmart-mcp-server.llama.svc.cluster.local:9001/sse"
POLICY_FILE = "data/return-policy.txt"

In [ ]:
# Initialize Llama Stack client
client = LlamaStackClient(base_url=LLAMA_STACK_URL)

## 3. Discover Available Models

In [ ]:
# List available models
models = client.models.list()
rich.print(models)

In [ ]:
# Extract LLM and embedding models
llm_model = next(m for m in models if m.model_type == "llm")
embedding_model = next(m for m in models if m.model_type == "embedding")

model_id = llm_model.identifier
embedding_model_id = embedding_model.identifier
embedding_dimension = int(embedding_model.metadata["embedding_dimension"])

print(f"LLM Model: {model_id}")
print(f"Embedding Model: {embedding_model_id}")
print(f"Embedding Dimension: {embedding_dimension}")

## 4. Setup RAG with Vector Store

This demonstrates **Llama Stack's RAG capabilities** using:
- FAISS vector store for semantic search
- Sentence transformers for embeddings
- File upload and chunking

In [ ]:
# Create vector store with FAISS
vector_store = client.vector_stores.create(
    name="techmart_policy_store",
    extra_body={
        "embedding_model": embedding_model_id,
        "embedding_dimension": embedding_dimension,
        "provider_id": "faiss",
    },
)

vector_store_id = vector_store.id
print(f"Created vector store: {vector_store_id}")

In [ ]:
# Upload return policy document
with open(POLICY_FILE, "rb") as f:
    file_info = client.files.create(
        file=("return-policy.txt", f),
        purpose="assistants",
    )

print(f"Uploaded file: {file_info.id}")

In [ ]:
# Add file to vector store with chunking strategy
vector_store_file = client.vector_stores.files.create(
    vector_store_id=vector_store_id,
    file_id=file_info.id,
    chunking_strategy={
        "type": "static",
        "static": {
            "max_chunk_size_tokens": 400,
            "chunk_overlap_tokens": 100,
        },
    },
)

rich.print(vector_store_file)

## 5. Test RAG - Policy Questions

Now we'll test the RAG system by asking questions about the return policy.

In [ ]:
# Test 1: General return policy question
query = "What is the return window for electronics?"

response = client.with_options(timeout=60.0).responses.create(
    model=model_id,
    input=query,
    instructions="You are a helpful customer service assistant. Use the retrieved context to answer questions about TechMart's return policy accurately and concisely.",
    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [vector_store_id],
        }
    ],
)

print("\n" + "="*80)
print(f"QUESTION: {query}")
print("="*80)
print(f"\nANSWER:\n{response.output_text}")
print("\n" + "="*80)

In [ ]:
# Test 2: Restocking fee question
query = "Are there any restocking fees for opened electronics?"

response = client.with_options(timeout=60.0).responses.create(
    model=model_id,
    input=query,
    instructions="You are a helpful customer service assistant. Use the retrieved context to answer questions about TechMart's return policy accurately and concisely.",
    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [vector_store_id],
        }
    ],
)

print("\n" + "="*80)
print(f"QUESTION: {query}")
print("="*80)
print(f"\nANSWER:\n{response.output_text}")
print("\n" + "="*80)

## 6. Test MCP Integration via Llama Stack

This demonstrates **MCP (Model Context Protocol)** integration through Llama Stack.
Llama Stack can directly call MCP servers as tools - no need for separate MCP client!

In [ ]:
# Test MCP: Get order details
# Llama Stack will automatically call the MCP server's get_order tool

query = "Get me the details for order ORD-2024-001"

response = client.with_options(timeout=60.0).responses.create(
    model=model_id,
    input=query,
    instructions="You are a helpful assistant. Use the available tools to answer the user's question.",
    tools=[
        {
            "type": "mcp",
            "server_label": "TechMartOrdersServer",
            "server_url": MCP_SERVER_URL,
        }
    ],
)

print("\n" + "="*80)
print(f"QUESTION: {query}")
print("="*80)
print(f"\nRESPONSE:\n{response.output_text}")
print("\n" + "="*80)
print("\nFull Response Object:")
rich.print(response)

In [ ]:
# Test MCP: Check return eligibility
# Llama Stack will automatically call the MCP server's check_return_eligibility tool

query = "Can I return order ORD-2024-003?"

response = client.with_options(timeout=60.0).responses.create(
    model=model_id,
    input=query,
    instructions="You are a helpful customer service assistant. Use the available tools to check return eligibility and provide a clear answer.",
    tools=[
        {
            "type": "mcp",
            "server_label": "TechMartOrdersServer",
            "server_url": MCP_SERVER_URL,
        }
    ],
)

print("\n" + "="*80)
print(f"QUESTION: {query}")
print("="*80)
print(f"\nRESPONSE:\n{response.output_text}")
print("\n" + "="*80)

## 7. RAG + MCP Integration

Now we combine both tools in a single request:
- **RAG**: Get policy information from vector store
- **MCP**: Get real-time order data
- **LLM**: Generate intelligent response combining both

In [ ]:
# Scenario 1: Customer asks about specific order return
# This will use BOTH RAG (policy) and MCP (order data)

query = "I want to return my order ORD-2024-001. What's the process and will I get a full refund?"

response = client.with_options(timeout=60.0).responses.create(
    model=model_id,
    input=query,
    instructions="""You are a helpful TechMart customer service assistant. 
    Use the MCP tools to get order details and check eligibility.
    Use the file_search tool to get relevant policy information.
    Provide a comprehensive answer combining both sources.""",
    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [vector_store_id],
        },
        {
            "type": "mcp",
            "server_label": "TechMartOrdersServer",
            "server_url": MCP_SERVER_URL,
        }
    ],
)

print("\n" + "="*80)
print("CUSTOMER QUESTION:")
print("="*80)
print(query)
print("\n" + "="*80)
print("ASSISTANT RESPONSE:")
print("="*80)
print(response.output_text)
print("\n" + "="*80)

In [ ]:
# Scenario 2: Electronics return with restocking fee question

query = "I have order ORD-2024-003 for a laptop. Can I return it and will there be a restocking fee?"

response = client.with_options(timeout=60.0).responses.create(
    model=model_id,
    input=query,
    instructions="""You are a helpful TechMart customer service assistant. 
    Use the MCP tools to get order details and check eligibility.
    Use the file_search tool to get relevant policy information about restocking fees.
    Provide a clear, detailed answer.""",
    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [vector_store_id],
        },
        {
            "type": "mcp",
            "server_label": "TechMartOrdersServer",
            "server_url": MCP_SERVER_URL,
        }
    ],
)

print("\n" + "="*80)
print("CUSTOMER QUESTION:")
print("="*80)
print(query)
print("\n" + "="*80)
print("ASSISTANT RESPONSE:")
print("="*80)
print(response.output_text)
print("\n" + "="*80)

In [ ]:
# Scenario 3: Multiple orders comparison

query = "I have two orders: ORD-2024-001 and ORD-2024-005. Which one can I return and why?"

response = client.with_options(timeout=600.0).responses.create(
    model=model_id,
    input=query,
    instructions="""You are a helpful TechMart customer service assistant. 
    Use the MCP tools to get details for both orders and check their eligibility.
    Use the file_search tool to explain the policy reasons.
    Compare both orders and explain clearly.""",
    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [vector_store_id],
        },
        {
            "type": "mcp",
            "server_label": "TechMartOrdersServer",
            "server_url": MCP_SERVER_URL,
        }
    ],
)

print("\n" + "="*80)
print("CUSTOMER QUESTION:")
print("="*80)
print(query)
print("\n" + "="*80)
print("ASSISTANT RESPONSE:")
print("="*80)
print(response.output_text)
print("\n" + "="*80)